## Chapter 3: Coding Attention Mechanisms

### 3.3 Attending to different parts of input with self-attention

#### 3.3.1 Simple self-attention mechanism without trainable weights

In [1]:
import torch

print(f"Torch version: {torch.__version__}")
print(f"GPU/CUDA available: {torch.cuda.is_available()}")

Torch version: 2.9.1+cu130
GPU/CUDA available: True


- Using attention mechanism, text generating decoder segment can selectively access different/all input tokens, implying certain input token have more significance in generating specific output token
- Self attention is a technique which is used to enhance input representation by enabling each position(token in that position) to engage and determine the relevance of every other position within the same sequence

- This section explains a very simplified variant of self-attention, which does not contain any trainable weights

- Suppose we are given an input sequence $x^{(1)}$ to $x^{(T)}$
  - The input is a text (for example, a sentence like "Your journey starts with one step") that has already been converted into token embeddings as described in chapter 2
  - For instance, $x^{(1)}$ is a d-dimensional vector representing the word "Your", and so forth
- **Goal:** compute context vectors $z^{(i)}$ for each input sequence element $x^{(i)}$ in $x^{(1)}$ to $x^{(T)}$ (where $z$ and $x$ have the same dimension)
    - A context vector $z^{(i)}$ is a weighted sum over the inputs $x^{(1)}$ to $x^{(T)}$
    - The context vector is "context"-specific to a certain input
      - Instead of $x^{(i)}$ as a placeholder for an arbitrary input token, let's consider the second input, $x^{(2)}$
      - And to continue with a concrete example, instead of the placeholder $z^{(i)}$, we consider the second output context vector, $z^{(2)}$
      - The second context vector, $z^{(2)}$, is a weighted sum over all inputs $x^{(1)}$ to $x^{(T)}$ weighted with respect to the second input element, $x^{(2)}$
      - The attention weights are the weights that determine how much each of the input elements contributes to the weighted sum when computing $z^{(2)}$
      - In short, think of $z^{(2)}$ as a modified version of $x^{(2)}$ that also incorporates information about all other input elements that are relevant to a given task at hand

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/07.webp" width="400px">

- (Please note that the numbers in this figure are truncated to one
digit after the decimal point to reduce visual clutter; similarly, other figures may also contain truncated values)

- By convention, the unnormalized attention weights are referred to as **"attention scores"** whereas the normalized attention scores, which sum to 1, are referred to as **"attention weights"**

<br>

- **Step 1:** compute unnormalized attention scores $\omega$
- Suppose we use the second input token as the query, that is, $q^{(2)} = x^{(2)}$, we compute the unnormalized attention scores via dot products:
    - $\omega_{21} = x^{(1)} q^{(2)\top}$
    - $\omega_{22} = x^{(2)} q^{(2)\top}$
    - $\omega_{23} = x^{(3)} q^{(2)\top}$
    - ...
    - $\omega_{2T} = x^{(T)} q^{(2)\top}$
- Above, $\omega$ is the Greek letter "omega" used to symbolize the unnormalized attention scores
    - The subscript "21" in $\omega_{21}$ means that input sequence element 2 was used as a query against input sequence element 1

In [2]:
inputs = torch.tensor([
    [0.43, 0.15, 0.89], # Your     (x^1)
    [0.55, 0.87, 0.66], # journey  (x^2)
    [0.57, 0.85, 0.64], # starts   (x^3)
    [0.22, 0.58, 0.33], # with     (x^4)
    [0.77, 0.25, 0.10], # one      (x^5)
    [0.05, 0.80, 0.55]  # step     (x^6)
])

print(f"Input tensor: {inputs}")

Input tensor: tensor([[0.4300, 0.1500, 0.8900],
        [0.5500, 0.8700, 0.6600],
        [0.5700, 0.8500, 0.6400],
        [0.2200, 0.5800, 0.3300],
        [0.7700, 0.2500, 0.1000],
        [0.0500, 0.8000, 0.5500]])


- The figure depicts the initial step in this process, which involves calculating the attention scores ω between $x^{(2)}$ and all other input elements through a dot product operation

<br>
<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/08.webp" width="400px">

In [3]:
query = inputs[1]       # 2nd token embedding in the input as query

# initialize empty tensor with n = number of rows in input(one attention score for each token)
attn_scores_2 = torch.empty(inputs.shape[0])
for idx, x_i in enumerate(inputs):
    attn_scores_2[idx] = torch.dot(x_i, query)

print(attn_scores_2)

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


In [4]:
# dot product is a simplified version of sum of element-wise matrix multiplication of two matrices
res = 0

i = 0   # 1st token 
for idx, x_i in enumerate(inputs[i]):
    res += inputs[i][idx] * query[idx]

print(f"Attention score of token 1 w.r.t token 2 is: {res}")

Attention score of token 1 w.r.t token 2 is: 0.9544000625610352


- **Step 2:** normalize the unnormalized attention scores ("omegas", $\omega$) so that they sum up to 1
- Here is a simple way to normalize the unnormalized attention scores to sum up to 1 (a convention, useful for interpretation, and important for training stability) 
- unnormalized weights might exponentially multiply along with input while training which causes instabilities and inefficiency in LLM training
<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/09.webp" width="500px">

In [5]:
# dividing each weight by the sum of all weights
tmp_attn_wgts_2 = attn_scores_2 / attn_scores_2.sum()

print(f"Attention weights: {tmp_attn_wgts_2}")
print(f"Sum of attentiona weights: {tmp_attn_wgts_2.sum()}")

Attention weights: tensor([0.1455, 0.2278, 0.2249, 0.1285, 0.1077, 0.1656])
Sum of attentiona weights: 1.0000001192092896


In [6]:
# using softmax function to normalize the attention scores
def softmax_naive(x):
    return torch.exp(x) / torch.exp(x).sum(dim=0)

attn_wgts_2_naive = softmax_naive(attn_scores_2)

print(f"Attention weights: {attn_wgts_2_naive}")
print(f"Sum of attentiona weights: {attn_wgts_2_naive.sum()}")

Attention weights: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum of attentiona weights: 1.0


In [7]:
attn_wgts_2 = torch.softmax(attn_scores_2, dim=0)

print(f"Attention weights: {attn_wgts_2}")
print(f"Sum of attentiona weights: {attn_wgts_2.sum()}")

Attention weights: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum of attentiona weights: 1.0


- **Step 3**: compute the context vector $z^{(2)}$ by multiplying the embedded input tokens, $x^{(i)}$ with the attention weights and sum the resulting vectors:
<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/10.webp" width="500px">

In [8]:
# initialize tensor with zeros
context_vec_2 = torch.zeros(query.shape)

for idx, x_i in enumerate(inputs):
    context_vec_2 += attn_wgts_2[idx] * x_i

print(f"Context vector of 2nd input token w.r.t 2nd token is: {context_vec_2}")

Context vector of 2nd input token w.r.t 2nd token is: tensor([0.4419, 0.6515, 0.5683])


#### 3.3.2 Generalizing simple self attention 
- Calculating attention weights w.r.t each tokens results in a **n x n** weight matrix where n is the number of tokens present in input
- Since attention is calculated against the input sentence/token itself, it is called self-attention

In [9]:
# iterating through all input tokens to use them as query and finding attention scores
attn_scores_fl = torch.empty(6, 6)     # 2D matrix to hold the scores, here n = 6

for i, x_i in enumerate(inputs):
    for j, x_j in enumerate(inputs):
        attn_scores_fl[i, j] = torch.dot(x_i, x_j)

print(f"The atention score matrix(using for loop) is: \n{attn_scores_fl}")

The atention score matrix(using for loop) is: 
tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


In [10]:
# matrix multiplication is efficient than for loops in pytorch 
attn_scores = inputs @ inputs.T     # another method => inputs.matmul(inputs.T)
print(f"The atention score matrix(using matrix multiplication) is: \n{attn_scores}")

The atention score matrix(using matrix multiplication) is: 
tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


**Note:** 
- The dimension in softmax function specified along which dimension softmax should be calculated:
    - dim=0: softmax is calculated along column wise where the sum of column element values is equal to one
    - dim=1: softmax is calculated along row wise where the sum of row element values is equal to one

In [11]:
# calculating attention weights using softmax
attn_weights = torch.softmax(attn_scores, dim=1)
print(f"Attention weights are: \n{attn_weights}")

Attention weights are: 
tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])


In [12]:
# calculating context vectors for all input tokens can be done using four for loops or using matrix multiplication
context_vectors = attn_weights @ inputs
# the context vector matrix contains information about the releavance/significance of each token in every other input token in the sequence
print(f"Context vectors for each input tokens are: \n{context_vectors}")

Context vectors for each input tokens are: 
tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])


In [13]:
# the context vector computation for self attention can be generalized as follows:
attn_scores = inputs @ inputs.T
attn_weights = torch.softmax(attn_scores, dim=1)
context_vectors = attn_weights @ inputs

print(f"Context vectors for each input tokens are: \n{context_vectors}")

Context vectors for each input tokens are: 
tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])


### 3.4 Implementing self-attention with trainable weights

#### 3.4.1 Computing attention weights step by step

- The self-attention mechanism used in original transformers is also called "scaled dot-product attention"
- The overall idea is similar to before:
  - We want to compute context vectors as weighted sums over the input vectors specific to a certain input element
  - For the above, we need attention weights
- The differences compared to the basic attention mechanism introduced earlier:
  - The most notable difference is the introduction of weight matrices that are updated during model training
  - These trainable weight matrices are crucial so that the model (specifically, the attention module inside the model) can learn to produce "good" context vectors


<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/14.webp" width="600px">

- Implementing the self-attention mechanism step by step, we will start by introducing the three training weight matrices $W_q$, $W_k$, and $W_v$
- These three matrices are used to project the embedded input tokens, $x^{(i)}$, into query, key, and value vectors via matrix multiplication:
  - Query vector: $q^{(i)} = x^{(i)}\,W_q $
  - Key vector: $k^{(i)} = x^{(i)}\,W_k $
  - Value vector: $v^{(i)} = x^{(i)}\,W_v $


- The embedding dimensions of the input $x$ and the query vector $q$ can be the same or different, depending on the model's design and specific implementation
- In GPT models, the input and output dimensions are usually the same, but for illustration purposes, to better follow the computation, we choose different input and output dimensions here

In [14]:
# calculating attention scores using query, key and value matrices w.r.t token input 2(index value 1)
x_2 = inputs[1]
dim_in = inputs.shape[1]        # input dimension is equal to no. of columns in each token
dim_out = 2                     # output dimension can be arbritary value or same based on implementation

In [15]:
# Initialize weight matrices
torch.manual_seed(123)

# The parameter method initiates the variable with trainable properties such as `requires_grad = True` even with torch.no_grad() method
W_query = torch.nn.Parameter(torch.rand(dim_in, dim_out))
W_keys = torch.nn.Parameter(torch.rand(dim_in, dim_out))
W_values = torch.nn.Parameter(torch.rand(dim_in, dim_out))

In [16]:
print(f"Weights matrices (q, K, V) are:\nW_q - {W_query}\nW_k - {W_keys}\nW_v - {W_values}")

Weights matrices (q, K, V) are:
W_q - Parameter containing:
tensor([[0.2961, 0.5166],
        [0.2517, 0.6886],
        [0.0740, 0.8665]], requires_grad=True)
W_k - Parameter containing:
tensor([[0.1366, 0.1025],
        [0.1841, 0.7264],
        [0.3153, 0.6871]], requires_grad=True)
W_v - Parameter containing:
tensor([[0.0756, 0.1966],
        [0.3164, 0.4017],
        [0.1186, 0.8274]], requires_grad=True)


In [17]:
# calculate values of query, Key and Value for input token 2
query_2 = x_2 @ W_query
key_2 = x_2 @ W_keys
value_2 = x_2 @ W_values

print(f"For input token 2:\nquery - {query_2}\nkey - {key_2}\nvalue - {value_2}")

For input token 2:
query - tensor([0.4306, 1.4551], grad_fn=<SqueezeBackward4>)
key - tensor([0.4433, 1.1419], grad_fn=<SqueezeBackward4>)
value - tensor([0.3951, 1.0037], grad_fn=<SqueezeBackward4>)


In [18]:
# calculating key and value matrix elements for all tokens
keys = inputs @ W_keys
values = inputs @ W_values

print(f"Key and value matrix(K, V) for each token are:\n"
      f"keys: \n{keys}\n"
      f"values: \n{values}"
    )

Key and value matrix(K, V) for each token are:
keys: 
tensor([[0.3669, 0.7646],
        [0.4433, 1.1419],
        [0.4361, 1.1156],
        [0.2408, 0.6706],
        [0.1827, 0.3292],
        [0.3275, 0.9642]], grad_fn=<MmBackward0>)
values: 
tensor([[0.1855, 0.8812],
        [0.3951, 1.0037],
        [0.3879, 0.9831],
        [0.2393, 0.5493],
        [0.1492, 0.3346],
        [0.3221, 0.7863]], grad_fn=<MmBackward0>)


- In the next step, **step 2**, we compute the unnormalized attention scores by computing the dot product between the query and each key vector:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/15.webp" width="600px">

In [19]:
# Calculating attention scores for input token 2
attn_scores_22 = query_2.dot(key_2)                         # or torch.dot(query_2,key_2)
print(f"Attention score omega_22 is: {attn_scores_22}")

Attention score omega_22 is: 1.8523844480514526


In [20]:
# calculating other attention scores for given query 2
attn_scores_2 = query_2 @ keys.T
print(f"Attention scores for all tokens w.r.t input token 2: {attn_scores_2}")

Attention scores for all tokens w.r.t input token 2: tensor([1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440],
       grad_fn=<SqueezeBackward4>)


- Next, in **step 3**, we compute the attention weights (normalized attention scores that sum up to 1) using the softmax function we used earlier
- The difference to earlier is that we now scale the attention scores by dividing them by the square root of the embedding dimension, $\sqrt{d_k}$ (i.e., `d_k**0.5`)
- Without this scaling(square root), attention scores can grow too large, especially when the embedding dimension is high. Large scores compress the softmax output toward extreme values (near 0 or 1), making gradients very small during backpropagation.

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/16.webp" width="600px">

In [21]:
dim_k = keys.shape[1]
attn_weights_2 = torch.softmax(attn_scores_2 / dim_k**0.5, dim=-1)

print(attn_weights_2)

tensor([0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820],
       grad_fn=<SoftmaxBackward0>)


<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/17.webp" width="600px">

- In **step 4**, we now compute the context vector for input query vector 2:

In [22]:
context_vector_2 = attn_weights_2 @ values
print(f"Context vector for input token 2 is: \n{context_vector_2}")

Context vector for input token 2 is: 
tensor([0.3061, 0.8210], grad_fn=<SqueezeBackward4>)


#### 3.4.2 Implementing compact self-attention class

In [23]:
from torch.nn import Module

class SelfAttention_v1(Module):
    def __init__(self, dim_in, dim_out):
        super().__init__()
        self.W_queries = torch.nn.Parameter(torch.rand(dim_in, dim_out))
        self.W_keys = torch.nn.Parameter(torch.rand(dim_in, dim_out))
        self.W_values = torch.nn.Parameter(torch.rand(dim_in, dim_out))
    
    def forward(self, x):
        # calculate the q,k,V values w.r.t each input token
        queries = x @ self.W_queries
        keys = x @ self.W_keys
        values = x @ self.W_values

        # calculate attention scores & weights w.r.t each input token
        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)

        # calculate context vectores for each input token
        context_vectors = attn_weights @ values
        return context_vectors


In [24]:
# using the self attention class to calculate context vectors
torch.manual_seed(123)

d_in = inputs.shape[1]
d_out = 2

sa_v1 = SelfAttention_v1(dim_in=d_in, dim_out=d_out)
sa_v1(inputs)

tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)

```python
# While calling a method, Python automatically binds the arguments you pass to the parameters defined in the method signature
# Call with inputs
# sa_v1(inputs)
#   ↓
# Python calls: sa_v1.__call__(inputs)
#   ↓
# Which internally calls: sa_v1.forward(inputs)
```

- The above selfAttention class uses *torch.rand()* to initiate weight matrices randomly which is not effective way of initializing weights for training when compared to the linear layer which does the same functionality
- Linear layer initiates weights along with bias values which can be disabled by setting it to false
- GPT 2 includes bias also for weights matrices of query, key and value while modern LLMs don't

In [25]:
class SelfAttention_v2(Module):
    def __init__(self, dim_in, dim_out, qkv_bias=False):
        super().__init__()
        self.W_queries = torch.nn.Linear(dim_in, dim_out, bias=qkv_bias)
        self.W_keys = torch.nn.Linear(dim_in, dim_out, bias=qkv_bias)
        self.W_values = torch.nn.Linear(dim_in, dim_out, bias=qkv_bias)
    
    def forward(self, x):
        # calculate the q,k,V values w.r.t each input token
        queries = self.W_queries(x)
        keys = self.W_keys(x)
        values = self.W_values(x)

        # calculate attention scores & weights w.r.t each input token
        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)

        # calculate context vectores for each input token
        context_vectors = attn_weights @ values
        return context_vectors

In [26]:
sa_v2 = SelfAttention_v2(dim_in=d_in, dim_out=d_out)
sa_v2(inputs)

tensor([[0.5085, 0.3508],
        [0.5084, 0.3508],
        [0.5084, 0.3506],
        [0.5074, 0.3471],
        [0.5076, 0.3446],
        [0.5077, 0.3493]], grad_fn=<MmBackward0>)

### 3.5 Hiding future tokens/values using causal attention

- In causal attention, the attention weights above the diagonal are masked, ensuring that for any given input, the LLM is unable to utilize future tokens while calculating the context vectors with the attention weight

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/19.webp" width="400px">

#### 3.5.1 Applying a causal attention mask

- Causal self-attention ensures that the model's prediction for a certain position in a sequence is only dependent on the known outputs at previous positions, not on future positions
- In simpler words, this ensures that each next word prediction should only depend on the preceding words
- To achieve this, for each given token, we mask out the future tokens (the ones that come after the current token in the input text)

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/20.webp" width="600px">

In [28]:
# Reusing the weight matrices from SelfAttention to implement a simple causal mask
queries = sa_v2.W_queries(inputs)
keys = sa_v2.W_keys(inputs)

attn_scores = queries @ keys.T
attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
print(f"Attention weights for all tokens w.r.t input tokens: \n{attn_weights}")

Attention weights for all tokens w.r.t input tokens: 
tensor([[0.1362, 0.1730, 0.1736, 0.1713, 0.1792, 0.1666],
        [0.1359, 0.1730, 0.1735, 0.1716, 0.1790, 0.1670],
        [0.1366, 0.1729, 0.1734, 0.1714, 0.1788, 0.1669],
        [0.1493, 0.1701, 0.1704, 0.1697, 0.1732, 0.1674],
        [0.1589, 0.1690, 0.1692, 0.1667, 0.1712, 0.1649],
        [0.1408, 0.1715, 0.1718, 0.1717, 0.1758, 0.1684]],
       grad_fn=<SoftmaxBackward0>)


In [35]:
# using simple mask to mask the elements above the diagonal
# torch.tril returns the lower triangular part of the matrix (2-D tensor) or batch of matrices input(elements below the diagonal value given, 0-main diagonal), the other elements of the result tensor out are set to 0.
simple_mask = torch.tril(torch.ones(attn_weights.shape), diagonal=0)
print(f"The simple mask created is: \n{simple_mask}")

The simple mask created is: 
tensor([[1., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1.]])


In [38]:
# using the mask to create a masked attention weight matrix
# multiply the attention weights with this mask to zero out the attention scores above the diagonal
masked_attn_weights = attn_weights * simple_mask
print(f"Masked attention weights are: \n{masked_attn_weights}")

Masked attention weights are: 
tensor([[0.1362, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1359, 0.1730, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1366, 0.1729, 0.1734, 0.0000, 0.0000, 0.0000],
        [0.1493, 0.1701, 0.1704, 0.1697, 0.0000, 0.0000],
        [0.1589, 0.1690, 0.1692, 0.1667, 0.1712, 0.0000],
        [0.1408, 0.1715, 0.1718, 0.1717, 0.1758, 0.1684]],
       grad_fn=<MulBackward0>)


- However, if the mask were applied after softmax, like above, it would disrupt the probability distribution created by softmax
- Softmax ensures that all output values sum to 1
- Masking after softmax would require re-normalizing the outputs to sum to 1 again, which complicates the process and might lead to unintended effects

- To make sure that the rows sum to 1, we can normalize the attention weights as follows:

In [43]:
row_sums = masked_attn_weights.sum(dim=-1, keepdim=True)
norm_masked_attn_weights = masked_attn_weights/row_sums
print(f"Normalized masked attention weights are: \n{norm_masked_attn_weights}")

Normalized masked attention weights are: 
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4400, 0.5600, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2830, 0.3580, 0.3590, 0.0000, 0.0000, 0.0000],
        [0.2264, 0.2579, 0.2583, 0.2574, 0.0000, 0.0000],
        [0.1903, 0.2024, 0.2026, 0.1997, 0.2051, 0.0000],
        [0.1408, 0.1715, 0.1718, 0.1717, 0.1758, 0.1684]],
       grad_fn=<DivBackward0>)


- Instead of zeroing out attention weights above the diagonal and renormalizing the results, we can mask the unnormalized attention scores above the diagonal with negative infinity before they enter the softmax function:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/21.webp" width="450px">

In [46]:
# applying mask to attention scores directly using negative infinity(since exp(-inf) = 0, thus 0 can be calculated in softmax)
triu_mask = torch.triu(torch.ones(attn_scores.shape), diagonal=1)
triu_mask

tensor([[0., 1., 1., 1., 1., 1.],
        [0., 0., 1., 1., 1., 1.],
        [0., 0., 0., 1., 1., 1.],
        [0., 0., 0., 0., 1., 1.],
        [0., 0., 0., 0., 0., 1.],
        [0., 0., 0., 0., 0., 0.]])

In [47]:
ninf_mask = triu_mask.masked_fill(triu_mask.bool(), -torch.inf)
ninf_mask

tensor([[0., -inf, -inf, -inf, -inf, -inf],
        [0., 0., -inf, -inf, -inf, -inf],
        [0., 0., 0., -inf, -inf, -inf],
        [0., 0., 0., 0., -inf, -inf],
        [0., 0., 0., 0., 0., -inf],
        [0., 0., 0., 0., 0., 0.]])

In [49]:
print(f"Attention scores are: \n{attn_scores}")

Attention scores are: 
tensor([[-0.2327,  0.1055,  0.1098,  0.0913,  0.1549,  0.0521],
        [-0.2396,  0.1015,  0.1057,  0.0902,  0.1501,  0.0518],
        [-0.2323,  0.1004,  0.1045,  0.0885,  0.1481,  0.0507],
        [-0.1344,  0.0502,  0.0523,  0.0470,  0.0753,  0.0272],
        [-0.0349,  0.0520,  0.0538,  0.0331,  0.0708,  0.0174],
        [-0.2142,  0.0650,  0.0679,  0.0668,  0.1004,  0.0395]],
       grad_fn=<MmBackward0>)


In [50]:
# similarly we can use the mask and fill the values where 1 is present in mask to attn_scores
masked_attn_scores = attn_scores.masked_fill(triu_mask.bool(), -torch.inf)
print(f"Masked attention scores are: \n{masked_attn_scores}")

Masked attention scores are: 
tensor([[-0.2327,    -inf,    -inf,    -inf,    -inf,    -inf],
        [-0.2396,  0.1015,    -inf,    -inf,    -inf,    -inf],
        [-0.2323,  0.1004,  0.1045,    -inf,    -inf,    -inf],
        [-0.1344,  0.0502,  0.0523,  0.0470,    -inf,    -inf],
        [-0.0349,  0.0520,  0.0538,  0.0331,  0.0708,    -inf],
        [-0.2142,  0.0650,  0.0679,  0.0668,  0.1004,  0.0395]],
       grad_fn=<MaskedFillBackward0>)


In [57]:
# apply softmax to normalize the masked attention scores which gives the masked attention weights
masked_attn_weights_m2 = torch.softmax(masked_attn_scores/keys.shape[-1]**0.5, dim=-1)
print(f"Normalized masked attention weights from method 1(3 steps) are: \n{norm_masked_attn_weights}\n")
print(f"Normalized masked attention weights from method 2(2 steps) are: \n{masked_attn_weights_m2}")

Normalized masked attention weights from method 1(3 steps) are: 
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4400, 0.5600, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2830, 0.3580, 0.3590, 0.0000, 0.0000, 0.0000],
        [0.2264, 0.2579, 0.2583, 0.2574, 0.0000, 0.0000],
        [0.1903, 0.2024, 0.2026, 0.1997, 0.2051, 0.0000],
        [0.1408, 0.1715, 0.1718, 0.1717, 0.1758, 0.1684]],
       grad_fn=<DivBackward0>)

Normalized masked attention weights from method 2(2 steps) are: 
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4400, 0.5600, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2830, 0.3580, 0.3590, 0.0000, 0.0000, 0.0000],
        [0.2264, 0.2579, 0.2583, 0.2574, 0.0000, 0.0000],
        [0.1903, 0.2024, 0.2026, 0.1997, 0.2051, 0.0000],
        [0.1408, 0.1715, 0.1718, 0.1717, 0.1758, 0.1684]],
       grad_fn=<SoftmaxBackward0>)


### 3.5.2 Masking additional attention weights with dropout

- In addition, we also apply dropout to reduce overfitting during training
- Dropout can be applied in several places:
  - for example, after computing the attention weights;
  - or after multiplying the attention weights with the value vectors
- Here, we will apply the dropout mask after computing the attention weights because it's more common

- Furthermore, in this specific example, we use a dropout rate of 50%, which means randomly masking out half of the attention weights. (When we train the GPT model later, we will use a lower dropout rate, such as 0.1 or 0.2)

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/22.webp" width="400px">

- If we apply a dropout rate of 0.5 (50%), the non-dropped values will be scaled accordingly by a factor of 1/0.5 = 2
- Scaling is done in order for the upcoming layers to get the same value as input both with and without the dropout step
- The scaling is calculated by the formula 1 / (1 - `dropout_rate`)

In [56]:
# torch is having a dropout layer predefined which can be used for creating dropout layer in LLM architecture
torch.manual_seed(123)

dropout = torch.nn.Dropout(p=0.5)
sample_tensor = torch.ones(6,6)

print(f"sample tensor before applying dropout: \n{sample_tensor}\n")
print(f"sample tensor after applying dropout: \n{dropout(sample_tensor)}")

sample tensor before applying dropout: 
tensor([[1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.]])

sample tensor after applying dropout: 
tensor([[2., 2., 0., 2., 2., 0.],
        [0., 0., 0., 2., 0., 2.],
        [2., 2., 2., 2., 0., 2.],
        [0., 2., 2., 0., 0., 2.],
        [0., 2., 0., 2., 0., 2.],
        [0., 2., 2., 2., 2., 0.]])


In [58]:
# applying dropout to attention weight matrix
dropout_attn_weights = dropout(masked_attn_weights_m2)
print(f"Attention weights after applying dropout are: \n{dropout_attn_weights}")

Attention weights after applying dropout are: 
tensor([[2.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.7160, 0.7181, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.5167, 0.5147, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.4101, 0.0000],
        [0.2815, 0.3430, 0.0000, 0.3434, 0.3516, 0.3368]],
       grad_fn=<MulBackward0>)


#### 3.5.3 Implementing Causal self-attention class

- generalizing implementation of causal attention class for batch of inputs which is output of dataloader instead of single input

In [72]:
# simple batch can be created by stacking the tensors using torch.stack
batch = torch.stack((inputs, inputs), dim=0)
print(f"batch with dimension {batch.shape} is: \n{batch}")

batch with dimension torch.Size([2, 6, 3]) is: 
tensor([[[0.4300, 0.1500, 0.8900],
         [0.5500, 0.8700, 0.6600],
         [0.5700, 0.8500, 0.6400],
         [0.2200, 0.5800, 0.3300],
         [0.7700, 0.2500, 0.1000],
         [0.0500, 0.8000, 0.5500]],

        [[0.4300, 0.1500, 0.8900],
         [0.5500, 0.8700, 0.6600],
         [0.5700, 0.8500, 0.6400],
         [0.2200, 0.5800, 0.3300],
         [0.7700, 0.2500, 0.1000],
         [0.0500, 0.8000, 0.5500]]])


In [64]:
class CausalAttention(torch.nn.Module):
    def __init__(self, dim_in, dim_out, total_tokens, dropout_rate=0.0, qkv_bias=False):
        super().__init__()
        self.W_queries = torch.nn.Linear(dim_in, dim_out, bias=qkv_bias)
        self.W_keys = torch.nn.Linear(dim_in, dim_out, bias=qkv_bias)
        self.W_values = torch.nn.Linear(dim_in, dim_out, bias=qkv_bias)
        # adding dropout layer and instantiating mask
        self.dropout_layer = torch.nn.Dropout(dropout_rate)
        # registr_buffer is used to register a buffer that should not be considered a model parameter which are arbitary values used for computation
        self.register_buffer('causal_mask', torch.triu(torch.ones(total_tokens, total_tokens), diagonal=1))
    
    def forward(self, x):
        # num_batch is the batch dimension which specifies number of input present in that batch
        # num_tokens is the number of tokens present in the given input chunk
        # dim_in is the input token embedding size
        # example x.shape => [2, 6, 3]
        #   - this batch has 2 input sentence
        #   - each input sentence has 6 tokens in it
        #   - each token is of shape (1, 3) => 1x3 matrix

        # For inputs where `num_tokens` exceeds `total_tokens`, this will result in errors in the mask creation further below.
        # In practice, this is not a problem since the LLM ensures that inputs do not exceed `total_tokens` before reaching this forward method.
        num_batch, num_tokens, dim_in = x.shape
        
        # calculate the q,k,V values w.r.t each input token
        queries = self.W_queries(x)
        keys = self.W_keys(x)
        values = self.W_values(x)

        # calculate attention scores & weights w.r.t each input token
        attn_scores = queries @ keys.transpose(1, 2)    # transpose for 2D+ tensor where tensor is transposed between the given indexed dimensions

        # apply causal mask to the attention scores inplace without a variable memory space used
        attn_scores.masked_fill_(   # _ operations in pytorch are inplace ops unlike their normal counterpart
            self.causal_mask.bool()[:num_tokens, :num_tokens], -torch.inf
        )
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)

        # apply dropout to the attention weights calculated
        attn_weights = self.dropout_layer(attn_weights)


        # calculate context vectores for each input token
        context_vectors = attn_weights @ values
        return context_vectors

In [74]:
torch.manual_seed(123)

n_tokens = batch.shape[1]   # dim-1 = 6 in [2, 6, 3]
causalAttn = CausalAttention(dim_in, dim_out, total_tokens=n_tokens, dropout_rate=0.0)
ca_context_vectors = causalAttn(batch)
print(f"context vectors from causal attention are: \n{ca_context_vectors}\n")
print(f"context vectors shape: ", ca_context_vectors.shape)

context vectors from causal attention are: 
tensor([[[-0.4519,  0.2216],
         [-0.5874,  0.0058],
         [-0.6300, -0.0632],
         [-0.5675, -0.0843],
         [-0.5526, -0.0981],
         [-0.5299, -0.1081]],

        [[-0.4519,  0.2216],
         [-0.5874,  0.0058],
         [-0.6300, -0.0632],
         [-0.5675, -0.0843],
         [-0.5526, -0.0981],
         [-0.5299, -0.1081]]], grad_fn=<UnsafeViewBackward0>)

context vectors shape:  torch.Size([2, 6, 2])


- Note: dropout is applied only during training and not in inference

### 3.6 Extending single-head attention to multi-head attention

#### 3.6.1 Stacking multiple single-head attention layers

- The main idea behind multi-head attention is to run the attention mechanism multiple times (in parallel) with different, learned linear projections. This allows the model to jointly attend to information from different representation subspaces at different positions.
- multi-head attention module can be obtained by stacking multiple single-head attention modules as below

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/25.webp" width="400px">

- Context vectors are obtained by concatinating the result of each single-head attention context vectors
 
<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/24.webp" width="400px">

In [ ]:
# multi-head attention can be implemented by creating n single-heads using loop

class MultiHeadAttentionWrapper(torch.nn.Module):
    def __init__(self, dim_in, dim_out, total_tokens, dropout_rate, num_heads, qkv_bias=False):
        super().__init__()
        self.heads = torch.nn.ModuleList([
            CausalAttention(dim_in, dim_out, total_tokens, dropout_rate, qkv_bias)
            for n in range(num_heads)
        ])
    
    def forward(self, x):
        return torch.cat([head(x) for head in self.heads], dim=-1)

In [ ]:
torch.manual_seed(123)

n_tokens = batch.shape[1] # This is the number of tokens
d_in, d_out = 3, 2
mhaw = MultiHeadAttentionWrapper(d_in, d_out, n_tokens, 0.0, num_heads=2)

mhaw_context_vectors = mhaw(batch)
print(f"context vectors from multi-head attention wrapper are: \n{mhaw_context_vectors}\n")
print(f"MHAW context vectors shape: ", mhaw_context_vectors.shape)

context vectors from multi-head attention wrapper are: 
tensor([[[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5874,  0.0058,  0.5891,  0.3257],
         [-0.6300, -0.0632,  0.6202,  0.3860],
         [-0.5675, -0.0843,  0.5478,  0.3589],
         [-0.5526, -0.0981,  0.5321,  0.3428],
         [-0.5299, -0.1081,  0.5077,  0.3493]],

        [[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5874,  0.0058,  0.5891,  0.3257],
         [-0.6300, -0.0632,  0.6202,  0.3860],
         [-0.5675, -0.0843,  0.5478,  0.3589],
         [-0.5526, -0.0981,  0.5321,  0.3428],
         [-0.5299, -0.1081,  0.5077,  0.3493]]], grad_fn=<CatBackward0>)

MHAW context vectors shape:  torch.Size([2, 6, 4])


- In the implementation above, the embedding dimension is 4, because we `d_out=2` as the embedding dimension for the key, query, and value vectors as well as the context vector. And since we have 2 attention heads, we have the output embedding dimension 2*2=4

#### 3.6.2 Implementing multi-head attention with weight splits

- While the above is an intuitive and fully functional implementation of multi-head attention (wrapping the single-head attention `CausalAttention` implementation from earlier), we can write a stand-alone class called `MultiHeadAttention` to achieve the same

- We don't concatenate single attention heads for this stand-alone `MultiHeadAttention` class
- Instead, we create single W_query, W_key, and W_value weight matrices and then split those into individual matrices for each attention head
- This implementation is efficient because the computation can be completed parallel for all heads in a single go instead of waiting for each head to compute sequentially while using for loop of single heads

In [85]:
class MultiHeadAttention(torch.nn.Module):
    def __init__(self, dim_in, dim_out, total_tokens, dropout_rate, num_heads, qkv_bias=False):
        super().__init__()
        assert(dim_out % num_heads == 0), "dim_out must be divisible by num_heads"

        # add the variables to self state that will be used later
        self.d_out = dim_out
        self.num_heads = num_heads
        self.head_dim = dim_out // num_heads    # spliting weight matrix using number of heads (ex: d_out = 4, heads = 2 => then each head will have ..x2 dimension) 

        self.W_queries = torch.nn.Linear(dim_in, dim_out, bias=qkv_bias)
        self.W_keys = torch.nn.Linear(dim_in, dim_out, bias=qkv_bias)
        self.W_values = torch.nn.Linear(dim_in, dim_out, bias=qkv_bias)
        self.projection = torch.nn.Linear(dim_out, dim_out)     # Linear layer to combine head outputs
        self.dropout_layer = torch.nn.Dropout(dropout_rate)
        # registr_buffer is used to register a buffer that should not be considered a model parameter which are arbitary values used for computation
        self.register_buffer('causal_mask', torch.triu(torch.ones(total_tokens, total_tokens), diagonal=1))
    
    def forward(self, x):
        num_batch, num_tokens, dim_in = x.shape
        
        # calculate the q,k,V values w.r.t each input token
        queries = self.W_queries(x)
        keys = self.W_keys(x)
        values = self.W_values(x)

        # implicitly split the matrix by adding a `num_heads` dimension
        # Unroll/reshape last dim: (num_batch, num_tokens, d_out) -> (num_batch, num_tokens, num_heads, head_dim)
        # example: d_out = 4 and num_heads = 2 => head_dim = 2(4/2)
        #   [0.43, 0.15, 0.89, 0.22]  ← Token 1
        #   [[0.43, 0.15],        ← Token 1, Head 0 (dims 0-1)
        #    [0.89, 0.22]]        ← Token 1, Head 1 (dims 2-3)
        queries = queries.view(num_batch, num_tokens, self.num_heads, self.head_dim)
        keys = keys.view(num_batch, num_tokens, self.num_heads, self.head_dim)
        values = values.view(num_batch, num_tokens, self.num_heads, self.head_dim)

        # Transpose: (num_batch, num_tokens, num_heads, head_dim) -> (num_batch, num_heads, num_tokens, head_dim)
        queries = queries.transpose(1, 2)
        keys = keys.transpose(1, 2)
        values = values.transpose(1, 2)

        # Compute scaled dot-product attention (aka self-attention) with a causal mask
        # Dot product for each head is computed by matrix multiplication of (num_tokens x head_dim) matrices
        attn_scores = queries @ keys.transpose(2, 3)

        # apply causal mask to the attention scores inplace without a variable memory space used
        attn_scores.masked_fill_(   # _ operations in pytorch are inplace ops unlike their normal counterpart
            self.causal_mask.bool()[:num_tokens, :num_tokens], -torch.inf
        )
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)

        # apply dropout to the attention weights calculated
        attn_weights = self.dropout_layer(attn_weights)

        # calculate context vectores for each input token
        context_vectors = (attn_weights @ values).transpose(1, 2)

        # Combine heads, where self.d_out = self.num_heads * self.head_dim
        context_vectors = context_vectors.contiguous().view(num_batch, num_tokens, self.d_out)
        context_vectors = self.projection(context_vectors)   # optional projection

        return context_vectors

In [88]:
torch.manual_seed(123)

n_batch, n_tokens, d_in = batch.shape
d_out = 2

mha = MultiHeadAttention(d_in, d_out, n_tokens, 0.0, num_heads=2)

mha_context_vectors = mha(batch)
print(f"context vectors from multi-head attention are: \n{mha_context_vectors}\n")
print(f"MHA context vectors shape: ", mha_context_vectors.shape)

context vectors from multi-head attention are: 
tensor([[[0.3190, 0.4858],
         [0.2943, 0.3897],
         [0.2856, 0.3593],
         [0.2693, 0.3873],
         [0.2639, 0.3928],
         [0.2575, 0.4028]],

        [[0.3190, 0.4858],
         [0.2943, 0.3897],
         [0.2856, 0.3593],
         [0.2693, 0.3873],
         [0.2639, 0.3928],
         [0.2575, 0.4028]]], grad_fn=<ViewBackward0>)

MHA context vectors shape:  torch.Size([2, 6, 2])


- Note that the above is essentially a rewritten version of `MultiHeadAttentionWrapper` that is more efficient
- The resulting output looks a bit different since the random weight initializations differ, but both are fully functional implementations

---

**A note about the output dimensions**

- In the `MultiHeadAttention` above, we used `d_out=2` to use the same setting as in the `MultiHeadAttentionWrapper` class earlier
- The `MultiHeadAttentionWrapper`, due the the concatenation, returns the output head dimension `d_out * num_heads` (i.e., `2*2 = 4`)
- However, the `MultiHeadAttention` class (to make it more user-friendly) allows us to control the output head dimension directly via `d_out`; this means, if we set `d_out = 2`, the output head dimension will be 2, regardless of the number of heads
- It may be more intuitive to use `MultiHeadAttention` with `d_out = 4` as pointed out [here](https://github.com/rasbt/LLMs-from-scratch/pull/859), so that it produces the same output dimensions as `MultiHeadAttentionWrapper` with `d_out = 2`.

---

- Note that in addition, we added a linear projection layer (`self.projection `) to the `MultiHeadAttention` class above. This is simply a linear transformation that doesn't change the dimensions. It's a standard convention to use such a projection layer in LLM implementation, but it's not strictly necessary (recent research has shown that it can be removed without affecting the modeling performance)

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/26.webp" width="400px">

In [89]:
# let's look at what happens when executing `attn_scores = queries @ keys.transpose(2, 3)`
# (b, num_heads, num_tokens, head_dim) = (1, 2, 3, 4)
a = torch.tensor([[[[0.2745, 0.6584, 0.2775, 0.8573],
                    [0.8993, 0.0390, 0.9268, 0.7388],
                    [0.7179, 0.7058, 0.9156, 0.4340]],

                   [[0.0772, 0.3565, 0.1479, 0.5331],
                    [0.4066, 0.2318, 0.4545, 0.9737],
                    [0.4606, 0.5159, 0.4220, 0.5786]]]])

result = a @ a.transpose(2, 3)
result

tensor([[[[1.3208, 1.1631, 1.2879],
          [1.1631, 2.2150, 1.8424],
          [1.2879, 1.8424, 2.0402]],

         [[0.4391, 0.7003, 0.5903],
          [0.7003, 1.3737, 1.0620],
          [0.5903, 1.0620, 0.9912]]]])

- In this case, the matrix multiplication implementation in PyTorch will handle the 4-dimensional input tensor so that the matrix multiplication is carried out between the 2 last dimensions (num_tokens, head_dim) and then repeated for the individual heads 


In [90]:
# following becomes a more compact way to compute the matrix multiplication for each head separately
first_head = a[0, 0, :, :]
first_res = first_head @ first_head.T
print("First head:\n", first_res)

second_head = a[0, 1, :, :]
second_res = second_head @ second_head.T
print("\nSecond head:\n", second_res)

First head:
 tensor([[1.3208, 1.1631, 1.2879],
        [1.1631, 2.2150, 1.8424],
        [1.2879, 1.8424, 2.0402]])

Second head:
 tensor([[0.4391, 0.7003, 0.5903],
        [0.7003, 1.3737, 1.0620],
        [0.5903, 1.0620, 0.9912]])


In [94]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

In [96]:
n_tokens = 1024     # context window - maximum tokens in a single input chunk
d_in, d_out = 768, 768
num_heads = 12

gpt_mha = MultiHeadAttention(d_in, d_out, n_tokens, 0.0, num_heads)

print(f"parameter count of multi-head attention in GPT2 is: {count_parameters(gpt_mha)}")

parameter count of multi-head attention in GPT2 is: 2360064
